# rues-linker — Notebook de PRODUCCIÓN (4 fuentes, dedup + record linkage)

**Versión:** v2.12.0 · **Modo:** disco (on-disk) por defecto · **Escala objetivo:** 2–4 M registros en Colab Free

---

## Qué hace este notebook

Toma **hasta 4 fuentes** de empresas (RUES, DIAN, CRM, IMPORTACIONES, …) que pueden
**traer NIT o no** (mezcla), y produce en un solo paso:

1. **Deduplicación** intra-fuente (varias filas de la misma empresa dentro de una fuente).
2. **Record linkage** cross-source (la misma empresa real apareciendo en varias fuentes).

El resultado es un **golden record por empresa** + una **tabla correlativa** que mapea
cada fila original a su grupo.

## Cómo está organizado (arquitectura de 2 celdas)

- **Celda EXTRAS** (correr una vez): imports, funciones, configuración de rutas. No tocar.
- **Celda EJECUTAR** (editar y correr): solo tus variables — rutas, nombres de fuentes,
  qué columnas tiene cada una. Es la única celda que modificas en el día a día.

## Régimen con NIT vs sin NIT — léelo antes de correr

| Régimen | Evidencia fuerte | Perfil recomendado | Confiabilidad |
|---|---|---|---|
| **Con NIT** (RUES, DIAN) | El NIT ancla la identidad | `produccion_estandar` | Alta (F1≈0.87, +ciudad ≈0.93) |
| **Sin NIT** (importaciones) | Solo nombre + ciudad | `deduplication_sin_nit_conservador` | **Media — sobre-fusiona** |

> ⚠️ **Sin NIT el sistema fusiona de más.** Para esas fuentes usa el perfil conservador
> (más grupos, menos falsos positivos). Ver el bloque de Corea más abajo.

## Paso 0 — Montar Drive e instalar la librería

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Extraer e instalar la librería desde el tarball en tu Drive
# Ajusta la ruta al tarball que subiste:
TARBALL = '/content/drive/MyDrive/rues-linker-v2.11.0.tar.gz'

import subprocess, sys
subprocess.run(['tar', '-xzf', TARBALL, '-C', '/content'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                '/content/rues-linker-v2.11.0[dev]'], check=True)

import record_linkage
print('✅ rues-linker', record_linkage.__version__, 'instalado')

## ===== CELDA EXTRAS — correr UNA vez (no editar en el día a día) =====

In [ ]:
# ===== SECCIÓN EXTRAS / UTILIDADES =====
from __future__ import annotations

import gc
import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import pandas as pd

from record_linkage.config.profiles import crear_config_orchestrator
from record_linkage.pipeline.orchestrator import Orchestrator

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False


@dataclass
class FuenteSpec:
    """Especificación de una fuente de datos.

    Args:
        nombre: identificador de la fuente (p. ej. 'RUES', 'IMPORTACIONES').
        ruta: ruta al archivo (.csv o .parquet) en disco.
        col_nit: nombre de la columna de NIT. '' si la fuente NO tiene NIT.
        col_name: nombre de la columna de razón social (requerido).
        col_ciudad: nombre de la columna de ciudad. None si no tiene.
    """

    nombre: str
    ruta: str
    col_name: str
    col_nit: str = ""
    col_ciudad: Optional[str] = None


@dataclass
class ConfigProduccion:
    """Configuración centralizada de una corrida de producción.

    Todo parámetro ajustable vive aquí — cero magic numbers en la lógica.
    """

    workspace: str
    nombre_corrida: str
    fuentes: list[FuenteSpec]
    perfil: str = "produccion_estandar"
    forzar_disco: bool = True
    peso_ciudad: float = 0.20
    ground_truth_path: Optional[str] = None
    gt_col_nit: str = "NIT"
    gt_col_name: str = "RAZON_SOCIAL"
    gt_col_group: str = "ID_GROUP"
    gt_col_ciudad: Optional[str] = "CIUDAD"

    def __post_init__(self) -> None:
        if not self.fuentes:
            raise ValueError("Debe especificar al menos una fuente.")
        if not 0.0 <= self.peso_ciudad <= 1.0:
            raise ValueError(f"peso_ciudad debe estar en [0,1], recibido {self.peso_ciudad}.")


def _ram_libre_gb() -> float:
    """RAM disponible en GB (0.0 si psutil no está)."""
    if _HAS_PSUTIL:
        return psutil.virtual_memory().available / 1024**3
    return 0.0


def _leer_fuente(spec: FuenteSpec) -> pd.DataFrame:
    """Lee una fuente desde disco y la normaliza al esquema interno.

    Renombra las columnas a NIT / RAZON_SOCIAL / CIUDAD, crea NIT vacío si la
    fuente no tiene, y marca la columna FUENTE. Valida existencia (Fail Fast).
    """
    ruta = Path(spec.ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"Fuente '{spec.nombre}': no existe {ruta}")

    if ruta.suffix == ".parquet":
        df = pd.read_parquet(ruta)
    else:
        df = pd.read_csv(ruta, dtype=str)

    if spec.col_name not in df.columns:
        raise ValueError(
            f"Fuente '{spec.nombre}': falta col_name '{spec.col_name}'. "
            f"Columnas: {list(df.columns)}"
        )

    out = pd.DataFrame()
    out["RAZON_SOCIAL"] = df[spec.col_name].astype(str)
    if spec.col_nit and spec.col_nit in df.columns:
        out["NIT"] = df[spec.col_nit].fillna("").astype(str)
    else:
        out["NIT"] = ""  # fuente sin NIT (régimen Corea)
    if spec.col_ciudad and spec.col_ciudad in df.columns:
        out["CIUDAD"] = df[spec.col_ciudad].fillna("").astype(str)
    out["FUENTE"] = spec.nombre

    pct_con_nit = (out["NIT"].str.len() > 0).mean() * 100
    print(f"  ✅ {spec.nombre:16} {len(out):>10,} filas · {pct_con_nit:5.1f}% con NIT")
    return out


def ejecutar_produccion(cfg: ConfigProduccion) -> dict:
    """Corre dedup + record linkage sobre las fuentes con el Orchestrator.

    Modo disco forzado por defecto (escala 2–4 M). Guarda golden, correlativa
    y metadata en {workspace}/{nombre_corrida}/.

    Returns:
        dict con 'golden', 'correlative', 'stats', 'out_dir'.
    """
    t0 = time.time()
    ws = Path(cfg.workspace)
    out_dir = ws / cfg.nombre_corrida
    out_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 64)
    print(f"PRODUCCIÓN · {cfg.nombre_corrida} · perfil={cfg.perfil}")
    print("=" * 64)
    print(f"RAM libre al inicio: {_ram_libre_gb():.1f} GB")
    print("\nFuentes:")

    # 1) Leer y normalizar todas las fuentes (Trust but Verify)
    sources = {spec.nombre: _leer_fuente(spec) for spec in cfg.fuentes}
    total = sum(len(d) for d in sources.values())
    print(f"\nTotal registros: {total:,}")

    # 2) Config del Orchestrator — disco forzado
    orch_cfg = crear_config_orchestrator(perfil=cfg.perfil, workspace=str(out_dir))
    if cfg.forzar_disco:
        orch_cfg["linkage_engine_class"] = "disk_based"
    print(f"Motor LSH: {orch_cfg.get('linkage_engine_class')}")

    # 3) Correr
    orch = Orchestrator(config=orch_cfg, sources=sources, work_dir=str(out_dir))
    resultado = orch.run()

    golden = resultado.get("golden")
    corr = resultado.get("correlative")

    # 4) Guardar en disco (parquet — eficiente para millones)
    if golden is not None:
        golden.to_parquet(out_dir / "golden_records.parquet", index=False)
    if corr is not None:
        corr.to_parquet(out_dir / "correlativa.parquet", index=False)

    elapsed = time.time() - t0
    stats = {
        "total_input": total,
        "golden_records": int(len(golden)) if golden is not None else None,
        "reduccion_pct": round((1 - len(golden) / total) * 100, 1)
        if golden is not None and total
        else None,
        "tiempo_seg": round(elapsed, 1),
        "ram_libre_fin_gb": round(_ram_libre_gb(), 1),
        "perfil": cfg.perfil,
        "motor": orch_cfg.get("linkage_engine_class"),
        "fuentes": {s.nombre: len(sources[s.nombre]) for s in cfg.fuentes},
    }
    with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 64)
    print("RESULTADO")
    print("=" * 64)
    print(f"  Entrada       : {total:>12,}")
    print(f"  Golden records: {stats['golden_records']:>12,}")
    print(f"  Reducción     : {stats['reduccion_pct']:>11}%")
    print(f"  Tiempo        : {stats['tiempo_seg']:>11}s")
    print(f"  Guardado en   : {out_dir}")
    print("=" * 64)

    return {"golden": golden, "correlative": corr, "stats": stats, "out_dir": out_dir}


def medir_contra_ground_truth(cfg: ConfigProduccion, usar_ciudad: bool = True) -> dict:
    """Mide F1/precision/recall del sistema contra una base de verdad.

    Esta es la LÍNEA BASE: el referente contra el que iteras. Corre el
    pipeline de deduplicación sobre el ground truth y compara los pares
    predichos contra los pares verdaderos (ID_GROUP).

    Returns:
        dict con 'f1', 'precision', 'recall', 'grupos_reales', 'grupos_pred'.
    """
    import io
    import contextlib

    from record_linkage.deduplication.unified import deduplicate_unified
    from record_linkage.evaluation.pairwise import evaluar_pares

    if not cfg.ground_truth_path:
        raise ValueError("cfg.ground_truth_path no está definido.")

    gt = pd.read_csv(cfg.ground_truth_path, dtype={cfg.gt_col_nit: str})
    gt[cfg.gt_col_nit] = gt[cfg.gt_col_nit].fillna("")
    gt["FUENTE"] = "GT"

    extra = None
    if usar_ciudad and cfg.gt_col_ciudad and cfg.gt_col_ciudad in gt.columns:
        extra = [
            {"column": cfg.gt_col_ciudad, "weight": cfg.peso_ciudad,
             "type": "token_set_ratio_signed"}
        ]

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        corr, _gold = deduplicate_unified(
            df_input=gt.copy(), col_nit=cfg.gt_col_nit, col_name=cfg.gt_col_name,
            mode="BALANCEADO", profile=cfg.perfil,
            output_dir=str(Path(cfg.workspace) / "_gt_baseline"),
            extra_features=extra,
        )
    corr = corr.sort_values("ORIGINAL_INDEX").reset_index(drop=True)
    res = evaluar_pares(gt[cfg.gt_col_group].to_numpy(), corr["ID_GRUPO"].to_numpy())

    out = {
        "f1": round(res.f1, 4), "precision": round(res.precision, 4),
        "recall": round(res.recall, 4),
        "grupos_reales": int(gt[cfg.gt_col_group].nunique()),
        "grupos_pred": int(corr["ID_GRUPO"].nunique()),
        "con_ciudad": extra is not None,
    }
    print("=" * 56)
    print(f"LÍNEA BASE contra {Path(cfg.ground_truth_path).name}")
    print("=" * 56)
    print(f"  F1        : {out['f1']}")
    print(f"  Precision : {out['precision']}")
    print(f"  Recall    : {out['recall']}")
    print(f"  Grupos    : {out['grupos_pred']} pred / {out['grupos_reales']} reales")
    print(f"  Con ciudad: {out['con_ciudad']}")
    print("=" * 56)
    return out


print('✅ Celda EXTRAS cargada. Funciones disponibles:')
print('   - ejecutar_produccion(cfg)         → corre dedup + linkage')
print('   - medir_contra_ground_truth(cfg)   → línea base de calidad')

## ===== CELDA EJECUTAR — edita solo esto =====

Define tus 4 fuentes y corre. **Cambia las rutas y los nombres de columnas** según tus archivos.

In [ ]:
# ── Limpieza de estado previo ──────────────────────────────────────
if 'resultado' in globals():
    del resultado
    gc.collect()

# ── Configuración ──────────────────────────────────────────────────
WORKSPACE = '/content/drive/MyDrive/rues-linker-produccion'

cfg = ConfigProduccion(
    workspace=WORKSPACE,
    nombre_corrida='corrida_4fuentes_v1',
    perfil='produccion_estandar',   # con NIT. Sin NIT → 'deduplication_sin_nit_conservador'
    forzar_disco=True,              # modo disco por defecto (2–4 M)
    peso_ciudad=0.20,
    fuentes=[
        # Ajusta rutas y nombres de columnas a TUS archivos:
        FuenteSpec('RUES',          f'{WORKSPACE}/input/rues.parquet',
                   col_name='RAZON_SOCIAL', col_nit='NIT', col_ciudad='CIUDAD'),
        FuenteSpec('DIAN',          f'{WORKSPACE}/input/dian.parquet',
                   col_name='RAZON_SOCIAL', col_nit='NIT', col_ciudad='CIUDAD'),
        FuenteSpec('CRM',           f'{WORKSPACE}/input/crm.parquet',
                   col_name='NOMBRE', col_nit='NIT'),
        # Fuente SIN NIT (régimen Corea) — solo nombre + ciudad:
        FuenteSpec('IMPORTACIONES', f'{WORKSPACE}/input/importaciones.parquet',
                   col_name='Razón Social Importador', col_ciudad='CIUDAD_DESTINATARIO'),
    ],
)

# ── Ejecución ──────────────────────────────────────────────────────
resultado = ejecutar_produccion(cfg)

# Inspeccionar los grupos más grandes (revisión cualitativa de sobre-fusión)
corr = resultado['correlative']
if corr is not None and 'ID_GRUPO' in corr.columns:
    sizes = corr.groupby('ID_GRUPO').size().sort_values(ascending=False)
    print('\nTop 5 grupos más grandes (revisa que no sobre-fusionen):')
    for gid in sizes.head(5).index:
        nombres = corr.loc[corr['ID_GRUPO'] == gid, 'RAZON_SOCIAL'].head(4).tolist()
        print(f'  Grupo {gid} (n={sizes[gid]}): {nombres}')

## Línea base contra la base de verdad

Mide F1/precision/recall del sistema contra un ground truth etiquetado. Es tu **referente**
para iterar: cada cambio de configuración debe compararse contra estos números.

In [ ]:
# Apunta a tu base de verdad (la que viene en el paquete o la tuya):
cfg_gt = ConfigProduccion(
    workspace=WORKSPACE,
    nombre_corrida='_baseline',
    perfil='deduplication_standard',
    fuentes=[FuenteSpec('GT', 'placeholder', col_name='RAZON_SOCIAL')],  # no se usa aquí
    ground_truth_path='/content/rues-linker-v2.11.0/tests/data/golden_truth_exhaustivo_ciudad.csv',
)

# Sin ciudad y con ciudad — para ver el aporte de la ciudad:
print('>>> SIN ciudad:')
base_sin = medir_contra_ground_truth(cfg_gt, usar_ciudad=False)
print('\n>>> CON ciudad:')
base_con = medir_contra_ground_truth(cfg_gt, usar_ciudad=True)

print(f"\nAporte de la ciudad al F1: {base_con['f1'] - base_sin['f1']:+.4f}")

## Caso especial — fuente SIN NIT (Corea): modo conservador + IDF

Cuando una fuente no trae NIT, el sistema **fusiona de más** porque el `token_set_ratio`
agrupa por tokens frecuentes compartidos ("ELITE EXPORTS... Y/O X", "X CORPORATION").

El perfil `deduplication_sin_nit_conservador` (v2.12.0) trae **re-scoring por IDF**:
pondera cada token por su rareza, así los tokens genéricos dejan de inflar la similitud.

> ⚠️ **Sobre la ciudad:** solo ayuda si es VARIADA. Si tu fuente está concentrada en
> pocas ciudades (caso Corea → casi todo SEOUL), la ciudad **empeora** la fusión.
> Para Corea, deja `PESO_CIUDAD = 0.0`. Para fuentes con ciudades variadas, prueba 0.20.

In [ ]:
import io, contextlib
from record_linkage.deduplication.unified import deduplicate_unified

# Cargar la fuente sin NIT (ej. el Excel de importaciones de Corea)
RUTA_SIN_NIT = '/content/drive/MyDrive/importaciones_corea.xlsx'
PESO_CIUDAD = 0.0   # 0.0 para Corea (todo SEOUL). 0.20 si la ciudad es variada.

df_imp = pd.read_excel(RUTA_SIN_NIT)
df_imp = df_imp.rename(columns={
    'Razón Social Importador': 'RAZON_SOCIAL',
    'CIUDAD_DESTINATARIO': 'CIUDAD',
})
df_imp['NIT'] = ''
df_imp['FUENTE'] = 'IMPORTACIONES'

extra = None
if PESO_CIUDAD > 0:
    extra = [{'column': 'CIUDAD', 'weight': PESO_CIUDAD, 'type': 'token_set_ratio_signed'}]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    corr, _gold = deduplicate_unified(
        df_input=df_imp.copy(), col_nit='NIT', col_name='RAZON_SOCIAL',
        mode='BALANCEADO', profile='deduplication_sin_nit_conservador',
        output_dir=f'{WORKSPACE}/sin_nit_corea', extra_features=extra,
    )

n = corr['ID_GRUPO'].nunique()
sizes = corr.groupby('ID_GRUPO').size().sort_values(ascending=False)
print(f'{len(df_imp)} registros sin NIT -> {n} grupos (reducción {(1-n/len(df_imp))*100:.1f}%)')
print(f'Grupo más grande: {sizes.max()}')
print('\nTop 8 grupos (revisa que NO mezclen empresas distintas):')
for g in sizes.head(8).index:
    nombres = corr.loc[corr['ID_GRUPO'] == g, 'RAZON_SOCIAL'].head(3).tolist()
    print(f'  Grupo {g} (n={sizes[g]}): {[x[:40] for x in nombres]}')

# Guardar
corr[['RAZON_SOCIAL', 'CIUDAD', 'ID_GRUPO', 'RAZON_SOCIAL_FINAL']].sort_values('ID_GRUPO').to_csv(
    f'{WORKSPACE}/correlativa_corea.csv', index=False)
print(f'\n✅ Guardado en {WORKSPACE}/correlativa_corea.csv')